# WalkProof D0 vision gate
Gemini Nano image input on Chrome 158 in a Kaggle free CPU session. Two configurations: stock Chrome, then Chrome with the multimodal input feature flag. Five fixture photos from the public repo, human answer key in fixtures/ground-truth.json.


In [ ]:

import json, os, subprocess, time

def sh(cmd, t=1800):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)
    print("$", cmd[:150])
    print((r.stdout or "").strip()[:1200])
    if (r.stderr or "").strip():
        print("STDERR:", r.stderr.strip()[-500:])
    print("[%.1fs]" % (time.time() - t0))
    return r

sh("nproc; grep MemTotal /proc/meminfo; df -h / /tmp /kaggle/working | tail -4")
sh("apt-get -qq update 2>&1 | tail -2")
sh("apt-get -qq install -y wget curl xvfb fonts-liberation libasound2t64 libatk1.0-0t64 "
   "libatk-bridge2.0-0t64 libxcomposite1 libxtst6 libnss3 libgbm1 2>&1 | tail -3")
sh("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /tmp/chrome.deb; ls -lh /tmp/chrome.deb")
sh("apt-get -qq install -y /tmp/chrome.deb 2>&1 | tail -3")
sh("google-chrome --version")
sh("pip -q install selenium 2>&1 | tail -2; python3 -c 'import selenium;print(selenium.__version__)'")

os.makedirs("/kaggle/working/fixtures", exist_ok=True)
BASE = "https://raw.githubusercontent.com/luke-shin-ai/walkproof/main/fixtures/"
for name in ["extinguisher.jpg", "exit-sign.jpg", "panel.jpg", "crack.jpg", "extinguisher-cart.jpg", "ground-truth.json"]:
    sh("curl -sSL -m 60 -o /kaggle/working/fixtures/" + name + " " + BASE + name)

gt = json.load(open("/kaggle/working/fixtures/ground-truth.json"))
print("fixtures:", [i["id"] for i in gt["items"]])


In [ ]:

import json, os, subprocess, sys, time

WORK = "/kaggle/working"
FIX = WORK + "/fixtures"
BASE = "https://raw.githubusercontent.com/luke-shin-ai/walkproof/main/fixtures/"
gt = json.load(open(FIX + "/ground-truth.json"))

SCHEMA = {
  "type": "object",
  "properties": {
    "targetPresent": {"type": "boolean"},
    "targetName": {"type": "string"},
    "defectType": {"type": "string", "enum": ["none", "blockage", "crack", "exposed_parts", "missing_label", "damaged_housing", "other"]},
    "severity": {"type": "string", "enum": ["none", "low", "medium", "high"]},
    "obscured": {"type": "boolean"},
    "note": {"type": "string"}
  },
  "required": ["targetPresent", "targetName", "defectType", "severity", "obscured", "note"]
}

def prompt_for(item):
    return ("You are checking one photo from a building safety walkthrough. "
            "Checklist item: " + item["prompt_item"] + ". "
            "What the inspector has to see: " + item["expects"] + ". "
            "Inspector note: none. "
            "Describe only what is visible in the photo. If the subject is not visible, set targetPresent to false. "
            "Set obscured to true when a cover, glare, or angle hides the part that matters. "
            "Use defectType none when nothing is wrong, and severity none in that case. "
            "Answer in the JSON object only.")

items = []
for entry in gt["items"]:
    items.append({"id": entry["id"], "url": BASE + entry["file"], "prompt": prompt_for(entry)})

HTML = r"""
<!doctype html><html><head><meta charset="utf-8"><title>walkproof vision gate</title>
<style>body{font-family:monospace;font-size:12px;margin:8px}pre{white-space:pre-wrap}</style></head>
<body><h3>WalkProof D0 vision gate</h3>
<button id="btn" style="font-size:16px;padding:6px 12px">run</button>
<pre id="log"></pre>
<script>
const ITEMS = __ITEMS__;
const SCHEMA = __SCHEMA__;
const OPTS = { expectedInputs: [{type:'text', languages:['en']}, {type:'image'}],
               expectedOutputs: [{type:'text', languages:['en']}] };
const logEl = document.getElementById('log');
const log = (m) => { logEl.textContent += m + "\n"; };
window.__log = () => logEl.textContent;
window.__result = null;
window.__avail = null;
const T0 = performance.now();
const stamp = () => "t+" + ((performance.now() - T0) / 1000).toFixed(1) + "s";
log(stamp() + " agent " + navigator.userAgent);
log("hardwareConcurrency " + navigator.hardwareConcurrency + " deviceMemory " + (navigator.deviceMemory || "n/a"));
log("LanguageModel " + typeof self.LanguageModel);
(async () => {
  if (typeof self.LanguageModel !== 'function') { log("NO_API"); return; }
  const avail = {};
  try {
    const t = performance.now();
    avail.image = await LanguageModel.availability(OPTS);
    log("availability_image " + avail.image + " " + (performance.now() - t).toFixed(0) + "ms");
  } catch (e) { avail.image = "error:" + e.name; log("availability_image_" + avail.image + " " + e.message); }
  try {
    avail.text = await LanguageModel.availability({ expectedInputs: [{type:'text', languages:['en']}],
                                                    expectedOutputs: [{type:'text', languages:['en']}] });
    log("availability_text " + avail.text);
  } catch (e) { avail.text = "error:" + e.name; log("availability_text_" + avail.text); }
  window.__avail = avail;
})();
document.getElementById('btn').onclick = async () => {
  const out = { images: [] };
  try {
    const tCreate = performance.now();
    let last = 0;
    const monitor = (m) => { m.addEventListener('downloadprogress', (e) => {
      const pct = Math.round(e.loaded * 100);
      if (pct - last >= 10) { last = pct; log(stamp() + " download " + pct + "%"); }
    }); };
    let session;
    try {
      session = await LanguageModel.create(Object.assign({}, OPTS, { monitor: monitor, responseConstraint: SCHEMA }));
      log("session_created constraint=yes");
    } catch (e1) {
      log("create_with_constraint_failed " + e1.name + " " + e1.message);
      session = await LanguageModel.create(Object.assign({}, OPTS, { monitor: monitor }));
      log("session_created constraint=no");
    }
    out.create_ms = Math.round(performance.now() - tCreate);
    log("SESSION_READY " + out.create_ms + "ms");
    for (const it of ITEMS) {
      const rec = { id: it.id, ms: null, text: "", json: null, err: null };
      const t = performance.now();
      try {
        const resp = await fetch(it.url);
        const blob = await resp.blob();
        const bmp = await createImageBitmap(blob);
        rec.bytes = blob.size;
        const input = [{type:'image', value: bmp}, {type:'text', value: it.prompt}];
        let text;
        try { text = await session.prompt(input, { responseConstraint: SCHEMA }); }
        catch (e2) { rec.prompt_constraint_error = e2.name + ":" + e2.message; text = await session.prompt(input); }
        rec.ms = Math.round(performance.now() - t);
        rec.text = String(text);
        try { rec.json = JSON.parse(rec.text.slice(rec.text.indexOf("{"), rec.text.lastIndexOf("}") + 1)); }
        catch (e3) { rec.json = null; }
        rec.input_usage = session.inputUsage;
        rec.input_quota = session.inputQuota;
      } catch (e) {
        rec.ms = Math.round(performance.now() - t);
        rec.err = e.name + ":" + e.message;
      }
      out.images.push(rec);
      log("IMG " + rec.id + " " + rec.ms + "ms json=" + (rec.json ? "ok" : "fail") + (rec.err ? " err=" + rec.err : ""));
      if (rec.json) log("JSON " + rec.id + " " + JSON.stringify(rec.json));
      else if (rec.text) log("RAW " + rec.id + " " + rec.text.slice(0, 400));
    }
    if (session.destroy) session.destroy();
    out.total_ms = Math.round(performance.now() - T0);
    window.__result = out;
    log("DONE " + JSON.stringify(out.images.map(function (r) {
      return { id: r.id, ms: r.ms, present: r.json && r.json.targetPresent, defect: r.json && r.json.defectType, severity: r.json && r.json.severity };
    })));
  } catch (e) {
    window.__result = { error: e.name + ":" + e.message, images: out.images };
    log("FATAL " + e.name + " " + e.message);
  }
};
</script></body></html>
"""

html = HTML.replace("__ITEMS__", json.dumps(items)).replace("__SCHEMA__", json.dumps(SCHEMA))
open(WORK + "/gate.html", "w").write(html)

srv = subprocess.Popen([sys.executable, "-m", "http.server", "8000", "--directory", WORK],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
xvfb = subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1600x1000x24"],
                        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.environ["DISPLAY"] = ":99"
time.sleep(2)

from selenium import webdriver
from selenium.webdriver.chrome.options import Options

PLAIN = []
MULTIMODAL = ["--enable-features=PromptApiForGeminiNanoMultimodalInput"]


def run(tag, extra, profile, timeout_s=900):
    opts = Options()
    opts.binary_location = "/usr/bin/google-chrome"
    opts.add_argument("--user-data-dir=" + profile)
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1400,900")
    for flag in extra:
        opts.add_argument(flag)
    t0 = time.time()
    driver = webdriver.Chrome(options=opts)
    print("=== [%s] chrome %s flags %s ===" % (tag, driver.capabilities.get("browserVersion"), extra), flush=True)
    driver.get("http://localhost:8000/gate.html")
    time.sleep(6)
    print("avail:", driver.execute_script("return window.__avail"), flush=True)
    print(driver.execute_script("return window.__log()"), flush=True)
    driver.find_element("id", "btn").click()
    deadline = time.time() + timeout_s
    last = ""
    while time.time() < deadline:
        time.sleep(15)
        text = driver.execute_script("return window.__log()")
        if text != last:
            print("--- t+%ds ---" % int(time.time() - t0), flush=True)
            print(text[-3000:], flush=True)
            last = text
        if "DONE " in text or "FATAL " in text:
            break
    result = driver.execute_script("return window.__result")
    avail = driver.execute_script("return window.__avail")
    print("=== RESULT %s ===" % tag, flush=True)
    print(json.dumps({"tag": tag, "flags": extra, "availability": avail, "result": result}, indent=1), flush=True)
    driver.quit()
    json.dump({"tag": tag, "flags": extra, "availability": avail, "result": result},
              open(WORK + "/gate_%s.json" % tag, "w"), indent=1)
    return {"tag": tag, "availability": avail, "result": result}


runs = []
runs.append(run("plain", PLAIN, "/tmp/profile-plain"))
runs.append(run("flag_multimodal", MULTIMODAL, "/tmp/profile-multimodal"))

print("=== SUMMARY ===", flush=True)
for entry in runs:
    res = entry["result"] or {}
    images = res.get("images") or []
    ok = [im for im in images if im.get("json")]
    print(entry["tag"], "avail", entry["availability"], "images", len(images), "json_ok", len(ok),
          "create_ms", res.get("create_ms"), "per_image_ms", [im.get("ms") for im in images])
print("du:", subprocess.run("du -sh /tmp/profile-plain /tmp/profile-multimodal 2>/dev/null", shell=True,
                            capture_output=True, text=True).stdout)
srv.terminate()
xvfb.terminate()
